# Tiền xử lý dữ liệu

---

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path
import os

DRIVE_WORKSPACE = Path('/content/drive/MyDrive/is353_variable_misuse')
DRIVE_REPO = DRIVE_WORKSPACE / 'detecting_variable_misuse_bugs'
DRIVE_RAW = DRIVE_WORKSPACE / 'dataset/raw/great'

LOCAL_REPO = Path('/content/detecting_variable_misuse_bugs')
LOCAL_WORK = Path('/content/great_work')
LOCAL_RAW = LOCAL_WORK / 'dataset/raw/great'
LOCAL_PROCESSED = LOCAL_WORK / 'dataset/processed/great_colab'
LOCAL_REPORT = LOCAL_PROCESSED / 'preprocessing_summary.json'

print('DRIVE_REPO:', DRIVE_REPO)
print('DRIVE_RAW:', DRIVE_RAW)
print('LOCAL_REPO:', LOCAL_REPO)
print('LOCAL_RAW:', LOCAL_RAW)
print('LOCAL_PROCESSED:', LOCAL_PROCESSED)

DRIVE_REPO: /content/drive/MyDrive/is353_variable_misuse/detecting_variable_misuse_bugs
DRIVE_RAW: /content/drive/MyDrive/is353_variable_misuse/dataset/raw/great
LOCAL_REPO: /content/detecting_variable_misuse_bugs
LOCAL_RAW: /content/great_work/dataset/raw/great
LOCAL_PROCESSED: /content/great_work/dataset/processed/great_colab


In [3]:
!python --version
!python -m pip --version
!df -h /content /content/drive
!nvidia-smi || true

Python 3.12.13
pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   44G   70G  39% /
drive           113G   47G   66G  42% /content/drive
Thu May 14 20:21:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
|

In [4]:
!ls -lh "$DRIVE_REPO"
!ls -lh "$DRIVE_REPO/scripts"
!test -f "$DRIVE_REPO/scripts/preprocess_great_data.py" && echo "OK: preprocess script exists"

!ls -lh "$DRIVE_RAW" | head
!find "$DRIVE_RAW" -maxdepth 2 -type f | wc -l

total 39K
-rw------- 1 root root  23K May 14 18:55 AGENTS.md
drwx------ 2 root root 4.0K May 14 18:55 docs
drwx------ 2 root root 4.0K May 14 18:55 scripts
drwx------ 2 root root 4.0K May 14 18:55 src
drwx------ 2 root root 4.0K May 14 18:55 tests
total 31K
-rw------- 1 root root  28K May 14 18:55 audit_great_dataset.py
-rw------- 1 root root 2.7K May 14 18:55 preprocess_great_data.py
OK: preprocess script exists
total 18K
drwx------ 2 root root 4.0K May 14 19:11 dev
drwx------ 2 root root 4.0K May 14 19:11 eval
-rw------- 1 root root  581 May 14 19:11 LICENSE
-rw------- 1 root root 4.4K May 14 19:11 README.md
drwx------ 2 root root 4.0K May 14 19:13 train
902


In [5]:
!rm -rf "$LOCAL_REPO"
!mkdir -p "$LOCAL_REPO"

!rsync -ah --info=progress2 \
  --exclude 'dataset/' \
  --exclude '.git/' \
  "$DRIVE_REPO/" \
  "$LOCAL_REPO/"

!ls -lh "$LOCAL_REPO/scripts"
!test -f "$LOCAL_REPO/scripts/preprocess_great_data.py" && echo "OK: local preprocess script exists"

          7.11M 100%  813.83kB/s    0:00:08 (xfr#40, to-chk=0/55)
total 32K
-rw------- 1 root root  28K May 14 18:55 audit_great_dataset.py
-rw------- 1 root root 2.7K May 14 18:55 preprocess_great_data.py
OK: local preprocess script exists


In [6]:
!rm -rf "$LOCAL_RAW"
!mkdir -p "$LOCAL_RAW"

!rsync -ah --info=progress2 \
  "$DRIVE_RAW/" \
  "$LOCAL_RAW/"

!find "$LOCAL_RAW" -maxdepth 2 -type f | wc -l
!du -sh "$LOCAL_RAW"
!df -h /content

         23.31G 100%   30.37MB/s    0:12:12 (xfr#902, to-chk=0/906)
902
22G	/content/great_work/dataset/raw/great
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   74G   39G  66% /


In [7]:
!rm -rf "$LOCAL_PROCESSED"

!PYTHONPATH="$LOCAL_REPO/src" \
python "$LOCAL_REPO/scripts/preprocess_great_data.py" \
  --input-root "$LOCAL_RAW" \
  --output-root "$LOCAL_PROCESSED" \
  --report-path "$LOCAL_REPORT" \
  --output-format compact

{
  "counters": {
    "read": 2952990,
    "written": 2884323,
    "dropped_too_many_tokens": 53068,
    "fixed_repair_target_added_to_candidates": 12666,
    "dropped_bug_sample_without_repair_target": 15598,
    "dropped_too_few_repair_candidates": 1
  },
  "class_counts": {
    "bug": 1434367,
    "no_bug": 1449956
  },
  "edge_group_counts": {
    "syntax": 79930821,
    "next_token": 554258762,
    "lexical": 36833980,
    "control_flow": 25268818,
    "data_flow": 81309475,
    "semantic": 130614
  },
  "license_counts": {
    "bsd-3-clause": 783565,
    "mit": 840222,
    "apache-2.0": 1056276,
    "bsd-2-clause": 161362,
    "isc": 12757,
    "gpl-3.0": 16356,
    "mpl-2.0": 3155,
    "cc0-1.0": 2047,
    "gpl-2.0": 6729,
    "epl-1.0": 726,
    "lgpl-3.0": 397,
    "lgpl-2.1": 593,
    "unlicense": 138
  }
}
preprocessing report: /content/great_work/dataset/processed/great_colab/preprocessing_summary.json


In [8]:
!ls -lh "$LOCAL_PROCESSED"
!wc -l "$LOCAL_PROCESSED"/*.jsonl
!du -sh "$LOCAL_PROCESSED"

import json
summary = json.loads(LOCAL_REPORT.read_text(encoding='utf-8'))
print(json.dumps(summary["aggregate"], indent=2, ensure_ascii=False))

total 16G
-rw-r--r-- 1 root root 1011M May 14 21:37 dev.jsonl
-rw-r--r-- 1 root root  5.2G May 14 22:01 eval.jsonl
-rw-r--r-- 1 root root  4.3K May 14 22:01 preprocessing_summary.json
-rw-r--r-- 1 root root  9.6G May 14 21:33 train.jsonl
     181478 /content/great_work/dataset/processed/great_colab/dev.jsonl
     946505 /content/great_work/dataset/processed/great_colab/eval.jsonl
    1756340 /content/great_work/dataset/processed/great_colab/train.jsonl
    2884323 total
16G	/content/great_work/dataset/processed/great_colab
{
  "counters": {
    "read": 2952990,
    "written": 2884323,
    "dropped_too_many_tokens": 53068,
    "fixed_repair_target_added_to_candidates": 12666,
    "dropped_bug_sample_without_repair_target": 15598,
    "dropped_too_few_repair_candidates": 1
  },
  "class_counts": {
    "bug": 1434367,
    "no_bug": 1449956
  },
  "edge_group_counts": {
    "syntax": 79930821,
    "next_token": 554258762,
    "lexical": 36833980,
    "control_flow": 25268818,
    "data_flo

In [9]:
ARCHIVE_PATH = Path('/content/great_preprocessed.zip')
PARTS_DIR = Path('/content/great_download_parts')

!rm -f "$ARCHIVE_PATH"
!rm -rf "$PARTS_DIR"
!mkdir -p "$PARTS_DIR"

!cd "$LOCAL_PROCESSED.parent" && zip -r "$ARCHIVE_PATH" "$LOCAL_PROCESSED.name"
!split -b 1900M "$ARCHIVE_PATH" "$PARTS_DIR/great_preprocessed.zip.part_"
!ls -lh "$PARTS_DIR"

  adding: great_colab/ (stored 0%)
  adding: great_colab/eval.jsonl (deflated 85%)
  adding: great_colab/preprocessing_summary.json (deflated 74%)
  adding: great_colab/train.jsonl (deflated 85%)
  adding: great_colab/dev.jsonl (deflated 85%)
total 2.4G
-rw-r--r-- 1 root root 1.9G May 14 22:15 great_preprocessed.zip.part_aa
-rw-r--r-- 1 root root 466M May 14 22:15 great_preprocessed.zip.part_ab


In [10]:
from google.colab import files

DOWNLOAD_NOW = False  # đổi thành True nếu muốn Colab tự mở download từng part

parts = sorted(PARTS_DIR.glob('great_preprocessed.zip.part_*'))
print('Parts:')
for part in parts:
    print(part)

if DOWNLOAD_NOW:
    for part in parts:
        files.download(str(part))
else:
    print("Bạn có thể tải thủ công trong panel Files ở /content/great_download_parts")

Parts:
/content/great_download_parts/great_preprocessed.zip.part_aa
/content/great_download_parts/great_preprocessed.zip.part_ab
Bạn có thể tải thủ công trong panel Files ở /content/great_download_parts
